# QSD Qiskit Simulation Suite

Interactive notebook for running all nine QSD quantum simulations on **local simulator** or **IBM Quantum hardware**.

---


## Cell 0 — Imports & environment check


In [ ]:
import sys, importlib

# If the suite is in a different directory, uncomment and adjust:
# sys.path.insert(0, "/path/to/qsd_suite/")

import qsd_qiskit_suite as qsd
# importlib.reload(qsd)  # uncomment if you edited the source mid-session

print("Environment:")
print(f"  Qiskit core …… {'✓' if qsd.QISKIT_AVAILABLE else '✗  pip install qiskit'}")
print(f"  Qiskit Aer ……… {'✓' if qsd.AER_AVAILABLE    else '✗  pip install qiskit-aer'}")
print(f"  IBM Runtime …… {'✓' if qsd.IBM_AVAILABLE     else '✗  pip install qiskit-ibm-runtime'}")
print()
print("Constants:")
print(f"  φ   = {qsd.PHI}")
print(f"  Hb₄ = {qsd.Hb4}")
print(f"  ς   = {qsd.GS}")
print(f"  μ₀  = {qsd.MU_0:.4f} GeV")
print(f"  E_syn = {qsd.E_SYN:.4f} GeV")
print(f"  α⁻¹×ς = {qsd.ALPHA_GS_PRODUCT:.10f}")


## Cell 1 — Backend configuration

**Run ONE of the options below** (comment/uncomment as needed).

| Option | When to use |
|--------|-------------|
| A | Quick local runs, no IBM account needed |
| B | Run on real IBM Quantum hardware |
| C | Hardware, auto-pick least-busy backend |


In [ ]:
# ─── OPTION A:  Local simulator (default) ───────────────────
qsd.configure_backend(
    use_hardware = False,
    shots        = 4096,
)

# ─── OPTION B:  Specific IBM backend ───────────────────────
# First time only — save credentials:
#
#   from qiskit_ibm_runtime import QiskitRuntimeService
#   QiskitRuntimeService.save_account(
#       channel="ibm_quantum", token="YOUR_TOKEN", set_as_default=True, overwrite=True)
#
# Then:
# qsd.configure_backend(
#     use_hardware = True,
#     backend_name = "ibm_brisbane",
#     shots        = 4096,
#     optimization_level = 3,
# )

# ─── OPTION C:  Auto-select least-busy IBM backend ────────
# qsd.configure_backend(use_hardware=True, auto_select=True, shots=4096)


---
## Cell 2 — Run ALL simulations at once


In [ ]:
results = qsd.run_all()


---
## Individual simulations

Run any single module below.  Each uses whichever backend was configured in Cell 1.


### 3a — Golden ratio from C₅


In [ ]:
gr = qsd.GoldenRatioCircuit()

# Analytical (always instant)
print(gr.eigenvalues_analytical())

# Estimator (runs on configured backend)
print(gr.run_estimator())

# Sampler (measurement circuit on configured backend)
print(gr.run_sampler())


### 3b — Holographic bit density Hb₄


In [ ]:
hb = qsd.HolographicBitDensity()
r  = hb.run()
print(f"Hb₄ = {r['Hb4']:.10f}")
print(f"Measured Shannon entropy  = {r['measured_entropy']:.4f}")
print(f"Expected n·Hb₄           = {r['expected_entropy']:.4f}")
print(f"Backend: {r['backend']}")


### 3c — K₄ Heisenberg exchange


In [ ]:
k4 = qsd.K4HeisenbergCircuit()

# Ground-state energy (Estimator → hardware-aware)
r = k4.measure_ground_state_energy()
print(f"Measured E₀ = {r['measured_energy']:.6f}")
print(f"Exact   E₀ = {r['exact_energy']:.6f}")
print(f"Expected    = −3J/2 = {r['expected_energy']:.6f}")


In [ ]:
# Trotter time-evolution sampling (Sampler → hardware-aware)
r = k4.run_sampler(n_trotter_steps=5, dt=0.2)
print(f"Counts (top 5): {dict(sorted(r['counts'].items(), key=lambda x: -x[1])[:5])}")
print(f"Backend: {r['backend']}")


### 3d — Mathieu potential


In [ ]:
mp = qsd.MathieuPotentialCircuit()
print(f"λ₁ = {mp.lambda1:.6f}")
print(f"E₀ (perturbative) = {mp.ground_state_energy():.6f}")


### 3e — Gauge force emergence (S₄ → SM)


In [ ]:
gf = qsd.GaugeForceEmergence()

# Analytical decomposition
print(gf.decompose_edge_representation())

# Run symmetry circuit on backend
r = gf.run_sampler()
print(f"Counts (top 5): {dict(sorted(r['counts'].items(), key=lambda x: -x[1])[:5])}")


### 3f — Radion–Higgs mixing


In [ ]:
rad = qsd.RadionScalarCircuit()
r   = rad.higgs_radion_mixing()
for k, v in r.items():
    print(f"  {k}: {v}")


### 3g — Energy dressing (μ₀ → E_syn)


In [ ]:
ed = qsd.EnergyDressingCircuit()
r  = ed.compute_dressed_energy()
print(f"Bare μ₀  = {r['E_bare_GeV']:.4f} GeV")
print(f"Dressed  = {r['E_dressed_GeV']:.4f} GeV  (factor 1+8ς = {r['dressing_factor']:.6f})")
print(f"CMS obs  = {r['CMS_observation']} GeV")


### 3h — 3-loop renormalization of α⁻¹


In [ ]:
rn = qsd.RenormalizationCircuit()
r  = rn.full_renormalization_result()
print(f"Tree:  {r['tree']:.3f}")
print(f"+Δ₁:   {r['1_loop']:.3f}")
print(f"−Δ₂:   {r['2_loop']:.3f}")
print(f"−Δ₃:   {r['3_loop']:.3f}")
print(f"────────────────────")
print(f"α⁻¹ = {r['alpha_inv_final']:.6f}  (exp: {r['experimental']:.6f},  {r['error_ppm']:.1f} ppm)")
print(f"α⁻¹ × ς = {r['alpha_inv_times_gs_EXACT']:.10f}  (≠ 2,  off by {r['deviation_from_2_percent']:.2f}%)")


### 3i — Syndrome prediction (28 GeV)


In [ ]:
sp = qsd.SyndromePredictionCircuit()

# Analytical prediction
r = sp.syndrome_energy_prediction()
print(f"E_syn = {r['E_syndrome_prediction_GeV']:.4f} GeV")
print(f"CMS   = {r['CMS_dimuon_excess_GeV']} GeV")
print(f"Agree = {r['agreement_percent']:.2f}%")


In [ ]:
# Run encoding circuit on backend
r = sp.run_sampler()
print(f"Counts (top 5): {dict(sorted(r['counts'].items(), key=lambda x: -x[1])[:5])}")


In [ ]:
# Run error-detection for each qubit
for eq in range(5):
    r = sp.run_error_detection(error_qubit=eq, error_type="X")
    top = max(r['counts'], key=r['counts'].get) if r['counts'] else "—"
    print(f"  X error on q{eq} → syndrome {top}")


---
## Cell 4 — Circuit diagrams


In [ ]:
circuits = qsd.create_example_circuits()

for name in ("c5_golden_ratio", "k4_ground_state", "syndrome_encode", "error_detection"):
    print(f"\n{'─'*40}\n{name}\n{'─'*40}")
    if qsd.QISKIT_AVAILABLE:
        print(circuits[name].draw(output='text', fold=100))
    else:
        print("  (install qiskit to see circuit diagrams)")


---
## Cell 5 — Switch to hardware mid-session

You can reconfigure at any time.  Subsequent `run_*` calls use the new backend.


In [ ]:
# Switch to hardware:
# qsd.configure_backend(use_hardware=True, backend_name="ibm_brisbane")

# Run one thing on hardware:
# r = qsd.SyndromePredictionCircuit().run_error_detection(error_qubit=2, error_type="X")
# print(r)

# Switch back to simulator:
# qsd.configure_backend(use_hardware=False)


---
## Cell 6 — Advanced: hardware job monitoring


In [ ]:
# from qiskit_ibm_runtime import QiskitRuntimeService
# service = QiskitRuntimeService()
#
# # Recent jobs:
# for j in service.jobs(limit=5):
#     print(f"  {j.job_id()}  {j.status()}  {j.backend().name}")
#
# # Retrieve a specific result:
# # job = service.job("JOB_ID")
# # result = job.result()


---
## Cell 7 — Advanced: custom backend options


In [ ]:
# Direct access to the BackendManager:
# qsd.backend.use_hardware = True
# qsd.backend.backend_name = "ibm_osaka"
# qsd.backend.shots = 8192
# qsd.backend.optimization_level = 2
# qsd.backend._backend = None   # force re-resolve on next call
#
# # List all available IBM backends:
# svc = qsd.backend._get_service()
# for b in svc.backends(operational=True, simulator=False):
#     print(f"  {b.name}  qubits={b.num_qubits}  pending={b.status().pending_jobs}")


---
## Quick reference

| What | How |
|------|-----|
| Switch to simulator | `qsd.configure_backend(use_hardware=False)` |
| Switch to hardware | `qsd.configure_backend(use_hardware=True, backend_name="ibm_brisbane")` |
| Auto-select backend | `qsd.configure_backend(use_hardware=True, auto_select=True)` |
| Change shots | `qsd.configure_backend(shots=8192)` or `qsd.backend.shots = 8192` |
| Run everything | `results = qsd.run_all()` |
| Run one simulation | `qsd.HolographicBitDensity().run()` |
| Get circuit diagram | `qsd.create_example_circuits()["k4_ground_state"].draw()` |
| Command-line sim | `python qsd_qiskit_suite.py` |
| Command-line HW | `python qsd_qiskit_suite.py --hw --backend ibm_brisbane` |
